# Imports & Configuration

In [ ]:
import arcpy
import os

# -------------------------------------------------------
# INPUT PATHS — update these
# -------------------------------------------------------
input_gdb  = r"C:\path\to\your\input.gdb"
output_gdb = r"C:\path\to\your\output.gdb"

civic_fc = os.path.join(input_gdb, "Civic_Points")   # Point feature class
road_fc  = os.path.join(input_gdb, "Road_Segments")  # Polyline feature class

# -------------------------------------------------------
# CIVIC FIELD NAMES — update to match your data
# -------------------------------------------------------
CIVIC_NAME_FIELD   = "StreetName"    # Text field — civic street name
CIVIC_NUMBER_FIELD = "StreetNumber"  # Numeric field — civic address number

# -------------------------------------------------------
# ROAD FIELD NAMES — update to match your data
# -------------------------------------------------------
ROAD_NAME_FIELD = "NAME_FULL"       # Text field — road full name
ROAD_FL_FIELD   = "F_Addr_L_911"    # From address, left side
ROAD_TL_FIELD   = "T_Addr_L_911"    # To address, left side
ROAD_FR_FIELD   = "F_Addr_R_911"    # From address, right side
ROAD_TR_FIELD   = "T_Addr_R_911"    # To address, right side

# -------------------------------------------------------
# OUTPUT PATHS
# -------------------------------------------------------
civic_result = os.path.join(output_gdb, "Civic_QA_Result")
output_lines = os.path.join(output_gdb, "Fishbone_Lines")
output_oor   = os.path.join(output_gdb, "Fishbone_OutOfRange")

sr = arcpy.Describe(civic_fc).spatialReference

print("Configuration loaded.")

# Copy Civic Layer

In [ ]:
print("Copying civic layer to output GDB...")

if arcpy.Exists(civic_result):
    arcpy.Delete_management(civic_result)

arcpy.CopyFeatures_management(civic_fc, civic_result)
arcpy.AddField_management(civic_result, "MatchedSegmentOID", "LONG")
arcpy.AddField_management(civic_result, "RangeStatus", "TEXT", field_length=20)

print("Civic layer copied and QA fields added.")

# Build Road Lookup Dictionaries

In [ ]:
print("Building road lookup dictionaries...")

road_fields = [
    "OBJECTID",
    ROAD_NAME_FIELD,
    ROAD_FL_FIELD, ROAD_TL_FIELD,
    ROAD_FR_FIELD, ROAD_TR_FIELD,
    "SHAPE@"
]

road_lookup        = {}
road_detail_lookup = {}

with arcpy.da.SearchCursor(road_fc, road_fields) as cur:
    for r in cur:
        oid  = r[0]
        name = r[1].upper().strip() if r[1] else None
        try:
            fL, tL = float(r[2]), float(r[3])
        except:
            fL = tL = None
        try:
            fR, tR = float(r[4]), float(r[5])
        except:
            fR = tR = None
        shp = r[6]

        if name:
            road_lookup.setdefault(name, []).append(
                (oid, fL, tL, fR, tR, r[1], shp)
            )
        road_detail_lookup[oid] = {
            "name_full": r[1],
            "fL": fL, "tL": tL,
            "fR": fR, "tR": tR,
            "shape": shp
        }

print(f"Road lookup built: {len(road_lookup)} unique street names.")
print(f"Road detail lookup built: {len(road_detail_lookup)} segments.")

# Match Civic Points to Road Segments

In [ ]:
print("Matching civic points to road segments...")

civic_fields = [
    "OBJECTID",
    CIVIC_NAME_FIELD,
    CIVIC_NUMBER_FIELD,
    "MatchedSegmentOID",
    "RangeStatus",
    "SHAPE@"
]

with arcpy.da.UpdateCursor(civic_result, civic_fields) as cur:
    for row in cur:
        street_num  = row[2]
        street_name = row[1].upper().strip() if row[1] else None

        if street_name is None or street_num is None:
            row[3] = None
            row[4] = "NoData"
            cur.updateRow(row)
            continue

        candidates = road_lookup.get(street_name, [])
        matched    = []

        for (oid, fL, tL, fR, tR, name_full, shp) in candidates:
            in_left  = fL is not None and tL is not None and min(fL, tL) <= street_num <= max(fL, tL)
            in_right = fR is not None and tR is not None and min(fR, tR) <= street_num <= max(fR, tR)
            if in_left or in_right:
                matched.append((oid, fL, tL, fR, tR, name_full, shp))

        if not matched:
            row[3] = None
            row[4] = "OutOfRange"
        else:
            civic_shp   = row[-1]
            min_dist    = None
            matched_oid = None
            for (oid, fL, tL, fR, tR, name_full, shp) in matched:
                dist = civic_shp.distanceTo(shp)
                if min_dist is None or dist < min_dist:
                    min_dist    = dist
                    matched_oid = oid
            row[3] = matched_oid
            row[4] = "WithinRange"

        cur.updateRow(row)

print("Matching complete.")

# Create Output Feature Classes

In [ ]:
# Fishbone_Lines
if arcpy.Exists(output_lines):
    arcpy.Delete_management(output_lines)

arcpy.CreateFeatureclass_management(
    out_path=os.path.dirname(output_lines),
    out_name=os.path.basename(output_lines),
    geometry_type="POLYLINE",
    spatial_reference=sr
)
for fname, ftype, flength in [
    ("CivicOID",     "LONG",   None),
    ("RoadOID",      "LONG",   None),
    ("RangeStatus",  "TEXT",   20),
    ("RoadName",     "TEXT",   100),
    ("CivicNumber",  "DOUBLE", None),
    ("MatchedRange", "TEXT",   50),
]:
    kwargs = {"field_length": flength} if flength else {}
    arcpy.AddField_management(output_lines, fname, ftype, **kwargs)

print("Fishbone_Lines created.")

# Fishbone_OutOfRange
if arcpy.Exists(output_oor):
    arcpy.Delete_management(output_oor)

arcpy.CreateFeatureclass_management(
    out_path=os.path.dirname(output_oor),
    out_name=os.path.basename(output_oor),
    geometry_type="POINT",
    spatial_reference=sr
)
for fname, ftype, flength in [
    ("CivicOID",    "LONG",   None),
    ("StreetName",  "TEXT",   100),
    ("CivicNumber", "DOUBLE", None),
    ("RangeStatus", "TEXT",   20),
]:
    kwargs = {"field_length": flength} if flength else {}
    arcpy.AddField_management(output_oor, fname, ftype, **kwargs)

print("Fishbone_OutOfRange created.")

# Pass 1: Draw Fishbone Lines

In [ ]:
print("Drawing fishbone lines...")

result_fields = [
    "OBJECTID", CIVIC_NAME_FIELD, CIVIC_NUMBER_FIELD,
    "SHAPE@", "MatchedSegmentOID", "RangeStatus"
]
line_fields = [
    "SHAPE@", "CivicOID", "RoadOID",
    "RangeStatus", "RoadName", "CivicNumber", "MatchedRange"
]

with arcpy.da.SearchCursor(civic_result, result_fields) as civic_cur, \
     arcpy.da.InsertCursor(output_lines, line_fields) as line_cur:

    for civ_oid, civ_street, civ_num, civ_shp, seg_oid, status in civic_cur:
        if civ_shp is None or status in ("OutOfRange", "NoData") or seg_oid is None:
            continue
        road = road_detail_lookup.get(seg_oid)
        if road is None:
            continue

        range_str = ""
        if road["fL"] is not None and road["tL"] is not None:
            lo, hi    = int(min(road["fL"], road["tL"])), int(max(road["fL"], road["tL"]))
            range_str = f"L:{lo}-{hi}"
        if road["fR"] is not None and road["tR"] is not None:
            lo, hi     = int(min(road["fR"], road["tR"])), int(max(road["fR"], road["tR"]))
            range_str += f" R:{lo}-{hi}" if range_str else f"R:{lo}-{hi}"

        civic_pt   = civ_shp.centroid
        road_shp   = road["shape"]
        nearest_pt = road_shp.positionAlongLine(road_shp.measureOnLine(civic_pt))
        line_geom  = arcpy.Polyline(
            arcpy.Array([civic_pt, nearest_pt.firstPoint]), sr
        )
        line_cur.insertRow([
            line_geom, civ_oid, seg_oid, status,
            road["name_full"], civ_num, range_str
        ])

print("Fishbone lines written.")

# Pass 2: Write OutOfRange Points

In [ ]:
print("Writing OutOfRange points...")

oor_fields = ["SHAPE@", "CivicOID", "StreetName", "CivicNumber", "RangeStatus"]

with arcpy.da.SearchCursor(civic_result, result_fields) as civic_cur, \
     arcpy.da.InsertCursor(output_oor, oor_fields) as oor_cur:

    for civ_oid, civ_street, civ_num, civ_shp, seg_oid, status in civic_cur:
        if civ_shp is None:
            continue
        if status in ("OutOfRange", "NoData"):
            oor_cur.insertRow([civ_shp, civ_oid, civ_street, civ_num, status])

print("OutOfRange points written.")

# Summary

In [ ]:
lines_count = int(arcpy.GetCount_management(output_lines)[0])
oor_count   = int(arcpy.GetCount_management(output_oor)[0])
civic_count = int(arcpy.GetCount_management(civic_result)[0])

print("=" * 50)
print("Fishbone QA Complete")
print("=" * 50)
print(f"  Total civic points processed : {civic_count}")
print(f"  Matched (WithinRange) lines  : {lines_count}")
print(f"  Unmatched (OutOfRange/NoData): {oor_count}")
print("-" * 50)
print(f"  Civic_QA_Result     → {civic_result}")
print(f"  Fishbone_Lines      → {output_lines}")
print(f"  Fishbone_OutOfRange → {output_oor}")
print("=" * 50)